# Notebook 16 — NumPy shape & cosine similarity

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate → Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `15-subprocess-and-archives.ipynb`

**Next up:** `17-pytest-fixtures-parametrize.ipynb`

---

## Learning objectives

- Reason about embedding tensors as `(batch, dim)` arrays.
- Normalize vectors and compute cosine similarity without Python loops over rows.
- Interpret broadcasting errors before scaling retrieval prototypes.

## Table of contents

1. **Install note (`numpy`)**
2. **Shapes, dtypes, norms**
3. **Vectorized cosine rows**
4. **Progressive drills — stack → normalize axis → top-k scores**
5. **Exercise — `cosine_scores`**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


### Dependency

Install once per environment:

```bash
pip install numpy
```

Colab runtimes ship **NumPy** by default.


## 1 · Shapes, dtypes, norms

*Explanation:* Embeddings arrive as **`float32`** matrices — `(num_candidates, dim)` — matching rows to a query vector is linear algebra, not nested lists.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
emb = rng.standard_normal((4, 8), dtype=np.float32)
query = rng.standard_normal((8,), dtype=np.float32)

print("emb", emb.shape, emb.dtype)
print("query norm", np.linalg.norm(query))


## 2 · Broadcasting warmup

*Explanation:* Subtract centroids or scale logits row-wise — broadcasting aligns trailing axes.


In [ ]:
import numpy as np

rows = np.arange(12, dtype=np.float32).reshape(3, 4)
offsets = np.array([10.0, 20.0, 30.0], dtype=np.float32).reshape(3, 1)
print(rows + offsets)


## 3 · Cosine similarity vectorized

*Explanation:* Cosine equals dot product after **unit normalization**. Doing `normalize(matrix, axis=1)` mirrors scoring every chunk vs one query.


In [ ]:
import numpy as np

rng = np.random.default_rng(1)
q = rng.standard_normal((128,))
mat = rng.standard_normal((256, 128))

qn = q / np.linalg.norm(q)
mn = mat / np.linalg.norm(mat, axis=1, keepdims=True)
scores = mn @ qn  # shape (256,)
print(scores.shape, float(scores.max()), float(scores.min()))


---

## Progressive drills — **easy → harder**

**Retrieval prototypes** almost always reduce to batched dot products — drills rehearse axis discipline.


### A · Easiest — batch dimension sanity

Know whether vectors are rows or columns before indexing GPU kernels mentally.


In [ ]:
import numpy as np

batch = np.ones((16, 768))
assert batch.ndim == 2
print(batch.shape[0], "vectors,", batch.shape[1], "dims")


### B · Medium — pairwise dot without loops

Mirror reranking **query × candidates**.


In [ ]:
import numpy as np

rng = np.random.default_rng(2)
q = rng.standard_normal((32,))
mat = rng.standard_normal((50, 32))
scores = mat @ q
print(scores.shape)


### C · Harder — stable softmax-shaped logits

Subtract row max before `exp` — classic numerical hygiene.


In [ ]:
import numpy as np

logits = np.array([[1000.0, 1001.0], [2.0, 3.0]], dtype=np.float64)
stable = logits - logits.max(axis=1, keepdims=True)
weights = np.exp(stable)
weights /= weights.sum(axis=1, keepdims=True)
print(weights.round(3))


### Exercise — `cosine_scores`

Implement **`cosine_scores(query, embeddings)`** where **`query`** has shape **`(dim,)`** and **`embeddings`** shape **`(n, dim)`**. Return a **`float64`** 1‑D array length **`n`** of cosine similarities. Treat zero vectors defensively by returning **`0.0`** for those rows (avoid divide‑by‑zero).


In [ ]:
import numpy as np


def cosine_scores(query: np.ndarray, embeddings: np.ndarray) -> np.ndarray:
    raise NotImplementedError


rng = np.random.default_rng(9)
q = rng.standard_normal((6,))
mat = rng.standard_normal((40, 6))

out = cosine_scores(q, mat)
assert out.shape == (40,)
qn = q / np.linalg.norm(q)
mn = mat / np.linalg.norm(mat, axis=1, keepdims=True)
expected = mn @ qn
mask = np.linalg.norm(mat, axis=1) == 0
expected = expected.astype(np.float64)
expected[mask] = 0.0
np.testing.assert_allclose(out, expected, rtol=1e-6)
print("OK")


### Solution — cosine_scores

<details>
<summary>Click to expand</summary>

```python
import numpy as np

def cosine_scores(query: np.ndarray, embeddings: np.ndarray) -> np.ndarray:
    q = np.asarray(query, dtype=np.float64)
    mat = np.asarray(embeddings, dtype=np.float64)
    row_norms = np.linalg.norm(mat, axis=1)
    q_norm = np.linalg.norm(q)
    out = np.zeros(mat.shape[0], dtype=np.float64)
    if q_norm == 0:
        return out
    mask = row_norms > 0
    q_unit = q / q_norm
    safe = mat[mask]
    safe_unit = safe / row_norms[mask][:, None]
    out[mask] = safe_unit @ q_unit
    return out
```

</details>
